# JOUR 2 - MODÉLISATION ET OPTIMISATION

**Projet Fil Rouge - Prédiction de Churn Client**

**Objectifs du Jour 2**:
- Comparer au minimum 5 algorithmes différents
- Optimiser les hyperparamètres
- Gérer le déséquilibre des classes
- Atteindre Recall ≥ 75%

**Durée estimée**: 7 heures

## 0. SETUP ET CHARGEMENT DES DONNÉES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, average_precision_score)

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report, make_scorer, f1_score

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, auc

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score)


from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [48]:
df = pd.read_csv('../data/archive/archivechurn_engineered.csv') 


X = df.drop(columns=['Churn','Churn_binary']) 
y = df['Churn'].astype(int) 

# 3. SPLIT TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



text_cols = X_train.select_dtypes(include=['object', 'string', 'bool']).columns

le = LabelEncoder()
for col in text_cols:
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

scaler = StandardScaler()
all_cols = X_train.columns

X_train[all_cols] = scaler.fit_transform(X_train[all_cols])
X_test[all_cols] = scaler.transform(X_test[all_cols])

print(f"Dimensions Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Reste-t-il du texte ? : {len(X_train.select_dtypes(include=['object']).columns) == 0}")

Dimensions Train: (2666, 37), Test: (667, 37)
Reste-t-il du texte ? : True


## PARTIE 2.1 - BENCHMARK DE MODÈLES (2h30)

### Mission 2.1.1 - Comparer 5-6 algorithmes

In [49]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, 
        class_weight='balanced', 
        random_state=42
    ),
    
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, 
        class_weight='balanced', 
        random_state=42
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100, 
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1
    ),
    
    'XGBoost': XGBClassifier(
        use_label_encoder=False, 
        eval_metric='logloss', 
        random_state=42,
        scale_pos_weight=5 
    ),
    
    'SVM': SVC(
        probability=True, 
        class_weight='balanced', 
        random_state=42
    ),

    # --- MODÈLES BONUS ---
    
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),

    'CatBoost': CatBoostClassifier(
        verbose=0, # Pour éviter de polluer l'affichage pendant l'entraînement
        random_state=42,
        auto_class_weights='Balanced'
    ),

    'Neural Network': MLPClassifier(
        hidden_layer_sizes=(64, 32), # Architecture standard
        max_iter=500,
        activation='relu',
        solver='adam',
        random_state=42
    )
}

print(f"Nombre de modèles à tester: {len(models)}")

Nombre de modèles à tester: 8


Logistic Regression : On l'utilise comme référence (baseline). Elle calcule une probabilité en traçant une ligne de séparation droite entre les clients qui partent et ceux qui restent.

Decision Tree : On le choisit pour sa simplicité. Il fonctionne par une série de questions binaires (si/alors) sur les données pour segmenter les clients en groupes homogènes.

Random Forest : On l'utilise pour sa robustesse. Il crée plusieurs arbres de décision indépendants et combine leurs résultats par vote pour réduire les erreurs de prédiction.

XGBoost : On le prend pour sa performance extrême. Il construit des arbres en série, où chaque nouvel arbre vient corriger spécifiquement les erreurs commises par les précédents.

SVM : On l'a choisi pour sa précision géométrique. Il cherche à maximiser la distance (la marge) entre les classes dans un espace à plusieurs dimensions pour isoler les churners.

KNN : On l'utilise pour son approche par voisinage. Il prédit le churn d'un client en regardant si les profils les plus similaires à lui dans la base de données sont déjà partis.

CatBoost : On l'a pris pour sa gestion des données texte. Il traite nativement les variables catégorielles sans préparation complexe et optimise le boosting pour éviter le sur-apprentissage.

Neural Network (MLP) : On l'utilise pour sa capacité d'abstraction. Il imite les connexions neuronales pour apprendre des relations mathématiques très complexes et cachées entre toutes les variables.

In [50]:
results = {}

for name, model in models.items():
    print(f"\nEntraînement de {name}...")
    
    # 1. Entraîner le modèle
    model.fit(X_train, y_train)
    
    # 2. Prédire les classes (y_pred) et les probabilités (y_proba)
    y_pred = model.predict(X_test)
    
    # Pour le calcul des AUC, on récupère la probabilité de la classe positive (1)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        # Cas particulier pour certains modèles sans predict_proba (ex: certains SVM sans probability=True)
        y_proba = model.decision_function(X_test)
    
    # 3. Calcul des métriques et stockage dans le dictionnaire results
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    }
    
    # Affichage rapide des performances clés (Objectif Recall >= 0.75)
    print(f"{name} - Recall: {results[name]['Recall']:.4f}, ROC-AUC: {results[name]['ROC-AUC']:.4f}")

print("\n--- Benchmark terminé ---")


Entraînement de Logistic Regression...
Logistic Regression - Recall: 0.7835, ROC-AUC: 0.8765

Entraînement de Decision Tree...
Decision Tree - Recall: 0.8557, ROC-AUC: 0.9185

Entraînement de Random Forest...
Random Forest - Recall: 0.8041, ROC-AUC: 0.9439

Entraînement de XGBoost...
XGBoost - Recall: 0.8866, ROC-AUC: 0.9439

Entraînement de SVM...
SVM - Recall: 0.7526, ROC-AUC: 0.9191

Entraînement de KNN...
KNN - Recall: 0.3918, ROC-AUC: 0.8462

Entraînement de CatBoost...
CatBoost - Recall: 0.8969, ROC-AUC: 0.9637

Entraînement de Neural Network...
Neural Network - Recall: 0.6907, ROC-AUC: 0.8928

--- Benchmark terminé ---


### Mission 2.1.2 - Tableau Comparatif

In [51]:
results_df = pd.DataFrame(results).T

# Tri croissant pour qu'en affichage horizontal, le meilleur soit en haut
results_df = results_df.sort_values('Recall', ascending=True)
results_df = results_df.round(4)


# Métriques dans l'ordre d'affichage
metrics = ['Recall', 'ROC-AUC', 'Precision', 'F1-Score']

print("\n RÉSULTATS DU BENCHMARK (Triés par Recall):")
display(results_df.sort_values('Recall', ascending=False))

# Définition des thèmes par modèle : [Nuance 1, Nuance 2, Nuance 3, Nuance 4]
# Ordre : du plus clair au plus sombre/intense pour chaque couleur
model_themes = {
    'Random Forest':       ['#B9F6CA', '#69F0AE', '#00E676', '#00C853'], # Verts
    'XGBoost':             ['#FF8A80', '#FF5252', '#FF1744', '#D50000'], # Rouges
    'CatBoost':            ['#FF80AB', '#FF4081', '#F50057', '#C51162'], # Roses
    'Logistic Regression': ['#80D8FF', '#40C4FF', '#00B0FF', '#0091EA'], # Bleus
    'Decision Tree':       ['#FFFF8D', '#FFFF00', '#FFEA00', '#FFD600'], # Jaunes
    'SVM':                 ['#EA80FC', '#E040FB', '#D500F9', '#AA00FF'], # Violets
    'KNN':                 ['#84FFFF', '#18FFFF', '#00E5FF', '#00B8D4'], # Cyans
    'Neural Network':      ['#FFD180', '#FFAB40', '#FF9100', '#FF6D00']  # Oranges
}

fig = go.Figure()

# Pour chaque métrique, on crée une barre
for i, metric in enumerate(metrics):
    
    # On récupère dynamiquement la couleur correspondante (i) pour chaque modèle (model)
    # dans l'ordre actuel du DataFrame (results_df.index)
    metric_colors = [model_themes[model][i] for model in results_df.index]
    
    fig.add_trace(go.Bar(
        name=metric,
        y=results_df.index,
        x=results_df[metric],
        orientation='h',
        marker=dict(
            color=metric_colors, # Application de la liste de couleurs
            line=dict(color='white', width=0.5)
        ),
        text=results_df[metric].apply(lambda x: f"{x:.2f}"),
        textposition='auto',
        textfont=dict(size=10, color='white')
    ))

# Ligne d'objectif métier (Recall 75%)
fig.add_vline(
    x=0.75, 
    line_dash="dash", 
    line_color="white", 
    annotation_text="Objectif Recall (0.75)", 
    annotation_position="bottom right",
    annotation_font_color="white"
)

# Mise en page globale
fig.update_layout(
    title="Comparaison des Modèles (Thèmes par Algorithme)",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    barmode='group', 
    height=800, 
    xaxis=dict(title="Score", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1.1]),
    yaxis=dict(title="", showgrid=False),
    legend=dict(
        title="Métriques :",
        orientation="h", 
        yanchor="bottom", y=1.02, 
        xanchor="right", x=1
    )
)

fig.show()


 RÉSULTATS DU BENCHMARK (Triés par Recall):


,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
CatBoost,0.9850,1.0000,0.8969,0.9457,0.9637,0.9342
XGBoost,0.9820,0.9885,0.8866,0.9348,0.9439,0.9251
Decision Tree,0.9400,0.7615,0.8557,0.8058,0.9185,0.8643
Random Forest,0.9715,1.0000,0.8041,0.8914,0.9439,0.9234
Logistic Regression,0.8036,0.4086,0.7835,0.5371,0.8765,0.6163
SVM,0.9070,0.6577,0.7526,0.7019,0.9191,0.8101
Neural Network,0.9340,0.8272,0.6907,0.7528,0.8928,0.7884
KNN,0.8996,0.8261,0.3918,0.5315,0.8462,0.6007


### Mission 2.1.3 - Analyse Comparative

**TODO**: Répondez aux questions suivantes:

1. **Quels sont les 3 meilleurs modèles selon le Recall ?**
   - 1er: CatBoost (Recall : 0.8969)
   - 2ème: XGBoost (Recall : 0.8866)
   - 3ème: Decision Tree (Recall : 0.8557)

2. **Quel modèle offre le meilleur compromis Precision/Recall ?**
   - CatBoost. Le meilleur indicateur du compromis Precision/Recall est le F1-Score. CatBoost possède le F1-Score le plus élevé (0.9457), car il réussit l'exploit de détecter 89.6% des churners (Recall) tout en ayant une précision parfaite de 100% sur le set de test (Precision = 1.0000 : tous les clients qu'il a prédits comme partants sont réellement partis). XGBoost est un excellent second avec un F1-Score de 0.9348

3. **Justifiez mathématiquement/théoriquement pourquoi certains modèles performent mieux:**
   - La supériorité des arbres boostés (CatBoost / XGBoost) : Le problème du Churn n'est pas linéaire (ex: c'est souvent la combinaison d'un plan international ET d'un grand nombre d'appels au SAV qui déclenche le départ). Les algorithmes d'ensemble basés sur les arbres découpent l'espace des données de manière non-linéaire. De plus, le Boosting corrige séquentiellement les erreurs : chaque nouvel arbre se concentre sur les clients difficiles à prédire (souvent la classe minoritaire des churners), ce qui explique leur Recall très élevé.
   - L'échec de l'approche géométrique (KNN) : Le KNN a le pire Recall (0.39). Théoriquement, il souffre de la "malédiction de la dimensionnalité" (curse of dimensionality). Avec tes 38 features (dimensions), la notion de "distance" entre deux clients devient floue, rendant la recherche des plus proches voisins très inefficace.
   - La limite du modèle linéaire (Logistic Regression) : Bien qu'il ait un Recall correct (0.78) grâce au class_weight='balanced', sa Precision est catastrophique (0.40). Mathématiquement, la régression logistique essaie de tracer un hyperplan (une ligne droite) pour séparer les classes. Le comportement humain de désabonnement étant complexe, tracer une simple ligne génère énormément de "Faux Positifs" (il prédit le churn pour beaucoup de clients fidèles).

### Mission 2.1.4 - Courbes ROC et PR

In [61]:
# 1. Sélection des 3 meilleurs modèles (basé sur notre analyse)
top_3_models = ['CatBoost', 'XGBoost', 'Decision Tree']

# Couleurs associées (pour garder la cohérence visuelle)
couleurs = {
     'CatBoost': "#408CFF",      
    'XGBoost': '#FF5252',       
    'Decision Tree': "#0DFF00"   
}

fig = go.Figure()

# 2. Boucle pour calculer et tracer la courbe de chaque modèle
for name in top_3_models:
    model = models[name]
    
    # Récupérer les probabilités de la classe positive (Churn = 1)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
        
    # Calcul du Taux de Faux Positifs (FPR) et Taux de Vrais Positifs (TPR)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    model_auc = auc(fpr, tpr)
    
    # Ajout de la courbe au graphique
    fig.add_trace(go.Scatter(
        x=fpr, 
        y=tpr,
        mode='lines',
        name=f'{name} (AUC = {model_auc:.4f})',
        line=dict(color=couleurs[name], width=3)
    ))

# 3. Ajout de la ligne de référence (Modèle Aléatoire)
fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[0, 1],
    mode='lines',
    name='Aléatoire (AUC = 0.5000)',
    line=dict(color='white', width=2, dash='dash')
))

# 4. Mise en page du graphique
fig.update_layout(
    title="Courbes ROC - Top 3 Modèles",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    xaxis=dict(title="Taux de Faux Positifs (FPR)", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1]),
    yaxis=dict(title="Taux de Vrais Positifs (TPR - Recall)", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1.05]),
    legend=dict(
        yanchor="bottom", y=0.05, 
        xanchor="right", x=0.95,
        bgcolor='rgba(0,0,0,0.5)',
        bordercolor='white',
        borderwidth=1
    ),
    width=800,
    height=600
)

fig.show()

In [ ]:



# 1. Sélection des 3 meilleurs modèles
top_3_models = ['CatBoost', 'XGBoost', 'Decision Tree']

# Couleurs associées
couleurs = {
    'CatBoost': "#408CFF",     
    'XGBoost': '#FF5252',       
    'Decision Tree': "#0DFF00"  
}

fig = go.Figure()

# 2. Ligne de base (Baseline) pour la courbe PR
# Contrairement à ROC (qui est toujours 0.5), la baseline PR dépend du pourcentage de la classe minoritaire (Churn)
taux_churn = y_test.mean()

# 3. Boucle pour calculer et tracer la courbe de chaque modèle
for name in top_3_models:
    model = models[name]
    
    # Récupérer les probabilités
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
        
    # Calcul de la précision et du recall pour différents seuils
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    # Ajout de la courbe au graphique
    fig.add_trace(go.Scatter(
        x=recall, 
        y=precision,
        mode='lines',
        name=f'{name} (PR-AUC = {pr_auc:.4f})',
        line=dict(color=couleurs[name], width=3)
    ))

# 4. Ajout de la ligne de référence (Modèle Aléatoire)
fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[taux_churn, taux_churn],
    mode='lines',
    name=f'Aléatoire (Baseline = {taux_churn:.4f})',
    line=dict(color='white', width=2, dash='dash')
))

# 5. Mise en page du graphique
fig.update_layout(
    title="Courbes Precision-Recall - Top 3 Modèles",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    xaxis=dict(title="Recall (Taux de détection des Churners)", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1.02]),
    yaxis=dict(title="Precision (Fiabilité des alertes)", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1.05]),
    legend=dict(
        yanchor="bottom", y=0.05, 
        xanchor="left", x=0.05, # On place la légende en bas à gauche pour ne pas cacher les courbes (souvent en haut à droite)
        bgcolor='rgba(0,0,0,0.5)',
        bordercolor='white',
        borderwidth=1
    ),
    width=800,
    height=600
)

fig.show()

## PARTIE 2.2 - GESTION DU DÉSÉQUILIBRE (1h30)

### Mission 2.2.1 - Tester 3 Stratégies de Rééquilibrage

In [58]:
# Sélectionner le meilleur modèle du benchmark pour tester les stratégies
# TODO: Instancier votre meilleur modèle
best_model_class = ...  # Ex: RandomForestClassifier
# D'après notre analyse (Recall de 0.89 et F1-Score de 0.94), CatBoost est le meilleur
best_model_class = CatBoostClassifier

print(f"Modèle sélectionné pour les tests de stratégies : {best_model_class.__name__}")

Modèle sélectionné pour les tests de stratégies : CatBoostClassifier


#### Stratégie 1: class_weight='balanced'

In [62]:

# Instanciation de CatBoost avec le poids équilibré
model_balanced = best_model_class(
    auto_class_weights='Balanced', # L'équivalent de class_weight='balanced' chez CatBoost
    random_state=42,
    verbose=0 # Pour cacher les logs d'entraînement
)

# Entraînement sur les données d'origine (sans SMOTE pour l'instant)
model_balanced.fit(X_train, y_train)

# Évaluation
y_pred_balanced = model_balanced.predict(X_test)

print("Stratégie 1 : Poids des classes équilibrés (auto_class_weights='Balanced')")
print("-" * 65)
print(classification_report(y_test, y_pred_balanced, target_names=['Fidèles (0)', 'Churners (1)']))

Stratégie 1 : Poids des classes équilibrés (auto_class_weights='Balanced')
-----------------------------------------------------------------
              precision    recall  f1-score   support

 Fidèles (0)       0.98      1.00      0.99       570
Churners (1)       1.00      0.90      0.95        97

    accuracy                           0.99       667
   macro avg       0.99      0.95      0.97       667
weighted avg       0.99      0.99      0.98       667



#### Stratégie 2: SMOTE

**Principe mathématique**: Générer des échantillons synthétiques de la classe minoritaire.

Formule: `x_new = x_i + λ × (x_neighbor - x_i)` où λ ∈ [0,1]

In [63]:


# TODO: Appliquer SMOTE sur le training set
smote = SMOTE(random_state=42)
# fit_resample va générer les données synthétiques pour équilibrer les classes
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribution des classes :")
print(f"Avant SMOTE:\n{y_train.value_counts().to_string()}")
print(f"\nAprès SMOTE:\n{y_train_smote.value_counts().to_string()}")

# Entraîner le modèle sur les nouvelles données équilibrées
# On utilise la classe sélectionnée précédemment (CatBoost) SANS le paramètre de poids
model_smote = best_model_class(
    random_state=42,
    verbose=0 # Pour cacher les logs d'entraînement si c'est CatBoost
)

model_smote.fit(X_train_smote, y_train_smote)

# Évaluation sur le set de test (qui lui, N'A PAS ÉTÉ MODIFIÉ, c'est très important !)
y_pred_smote = model_smote.predict(X_test)

print("\n" + "="*55)
print("Stratégie 2 : SMOTE (Sur-échantillonnage)")
print("="*55)
print(classification_report(y_test, y_pred_smote, target_names=['Fidèles (0)', 'Churners (1)']))

Distribution des classes :
Avant SMOTE:
Churn
0    2280
1     386

Après SMOTE:
Churn
0    2280
1    2280

Stratégie 2 : SMOTE (Sur-échantillonnage)
              precision    recall  f1-score   support

 Fidèles (0)       0.98      1.00      0.99       570
Churners (1)       1.00      0.88      0.93        97

    accuracy                           0.98       667
   macro avg       0.99      0.94      0.96       667
weighted avg       0.98      0.98      0.98       667



#### Stratégie 3: Undersampling

In [64]:
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("Distribution des classes :")
print(f"Après Undersampling:\n{y_train_rus.value_counts().to_string()}")

# Entraîner le modèle sur les nouvelles données réduites
# Toujours sans le paramètre de poids (auto_class_weights) car les classes sont maintenant équilibrées
model_rus = best_model_class(
    random_state=42,
    verbose=0 # Pour cacher les logs d'entraînement
)

model_rus.fit(X_train_rus, y_train_rus)

# Évaluation sur le set de test
y_pred_rus = model_rus.predict(X_test)

print("\n" + "="*55)
print("Stratégie 3: Undersampling (RandomUnderSampler)")
print("="*55)
print(classification_report(y_test, y_pred_rus, target_names=['Fidèles (0)', 'Churners (1)']))

Distribution des classes :
Après Undersampling:
Churn
0    386
1    386

Stratégie 3: Undersampling (RandomUnderSampler)
              precision    recall  f1-score   support

 Fidèles (0)       0.98      0.95      0.97       570
Churners (1)       0.77      0.90      0.83        97

    accuracy                           0.95       667
   macro avg       0.88      0.93      0.90       667
weighted avg       0.95      0.95      0.95       667



### Mission 2.2.2 - Comparaison des Stratégies

In [65]:
strategies_results = {
    'No Rebalancing': {
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    },
    'class_weight': {
        'Recall': recall_score(y_test, y_pred_balanced),
        'Precision': precision_score(y_test, y_pred_balanced),
        'F1-Score': f1_score(y_test, y_pred_balanced)
    },
    'SMOTE': {
        'Recall': recall_score(y_test, y_pred_smote),
        'Precision': precision_score(y_test, y_pred_smote),
        'F1-Score': f1_score(y_test, y_pred_smote)
    },
    'Undersampling': {
        'Recall': recall_score(y_test, y_pred_rus),
        'Precision': precision_score(y_test, y_pred_rus),
        'F1-Score': f1_score(y_test, y_pred_rus)
    }
}

# Création et affichage du DataFrame
strategies_df = pd.DataFrame(strategies_results).T
strategies_df = strategies_df.round(4)

print(" RÉSULTATS DES STRATÉGIES DE RÉÉQUILIBRAGE:")
display(strategies_df)

# ===================================================================
# 2. VISUALISATION INTERACTIVE (PLOTLY)
# ===================================================================

metrics = ['Recall', 'Precision', 'F1-Score']
# Couleurs : Orange (Recall), Cyan (Precision), Vert (F1-Score)
colors = ['#FF4500', '#00FFFF', '#00FF7F'] 

fig = go.Figure()

for i, metric in enumerate(metrics):
    fig.add_trace(go.Bar(
        name=metric,
        x=strategies_df.index,
        y=strategies_df[metric],
        marker=dict(
            color=colors[i],
            line=dict(color='white', width=0.5)
        ),
        text=strategies_df[metric].apply(lambda x: f"{x:.2f}"),
        textposition='auto',
        textfont=dict(size=12, color='white')
    ))

# Ajout d'une ligne d'objectif métier (Recall > 0.75)
fig.add_hline(
    y=0.75, 
    line_dash="dash", 
    line_color="yellow", 
    annotation_text="Objectif Recall (0.75)", 
    annotation_position="top left",
    annotation_font_color="yellow"
)

# Mise en page du graphique
fig.update_layout(
    title="Comparaison des Stratégies de Gestion du Déséquilibre",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    barmode='group', # Affiche les barres côte à côte pour chaque stratégie
    xaxis=dict(title="Stratégie Testée", showgrid=False),
    yaxis=dict(title="Score", showgrid=True, gridcolor='rgba(255,255,255,0.1)', range=[0, 1.15]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=600
)

fig.show()

 RÉSULTATS DES STRATÉGIES DE RÉÉQUILIBRAGE:


,Recall,Precision,F1-Score
No Rebalancing,0.6907,0.8272,0.7528
class_weight,0.8969,1.0000,0.9457
SMOTE,0.8763,1.0000,0.9341
Undersampling,0.8969,0.7699,0.8286


**Question**: Quelle stratégie sélectionnez-vous et pourquoi ?

**Réponse**: Je sélectionne la stratégie class_weight (Ajustement du poids des classes).



Performances mathématiques optimales : C'est la stratégie qui offre le meilleur compromis global avec le F1-Score le plus élevé (0.9457). Elle réussit l'exploit d'atteindre le meilleur Recall (0.8969) (ex æquo avec l'Undersampling) tout en maintenant une Precision parfaite (1.0000).

Impact métier idéal : Avec une précision de 100%, nous avons la certitude que chaque client ciblé par le modèle est un vrai "churner". Le budget marketing de rétention (promotions, appels) sera donc dépensé avec une efficacité absolue, sans déranger inutilement les clients fidèles (zéro faux positif).

Simplicité et robustesse (MLOps) : Contrairement à SMOTE (qui invente des données synthétiques au risque de créer du sur-apprentissage) ou à l'Undersampling (qui détruit de précieuses données sur les clients fidèles), class_weight ne modifie pas le dataset d'origine. Il modifie simplement la fonction de coût de l'algorithme. C'est une approche plus légère, plus rapide à entraîner et beaucoup plus facile à mettre en production.

## PARTIE 2.3 - OPTIMISATION AVANCÉE (2h)

### Mission 2.3.1 - GridSearchCV ou RandomizedSearchCV

In [67]:

param_grid = {
    'iterations': [100, 300, 500],           # Nombre d'arbres (équivalent à n_estimators)
    'depth': [4, 6, 8],                      # Profondeur de chaque arbre (éviter > 10 pour le sur-apprentissage)
    'learning_rate': [0.01, 0.05, 0.1],      # Vitesse d'apprentissage (plus c'est bas, plus il faut d'itérations)
    'l2_leaf_reg': [1, 3, 5, 9],             # Régularisation L2 (pour pénaliser les arbres trop complexes)
    'auto_class_weights': ['Balanced']       # 🏆 NOTRE STRATÉGIE GAGNANTE DE L'ÉTAPE PRÉCÉDENTE
}

print(f"Grille de paramètres définie pour CatBoost avec {len(param_grid['iterations']) * len(param_grid['depth']) * len(param_grid['learning_rate']) * len(param_grid['l2_leaf_reg'])} combinaisons possibles.")

Grille de paramètres définie pour CatBoost avec 108 combinaisons possibles.


In [68]:
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier

# Instanciation du modèle de base (avec notre stratégie de poids équilibrés)
base_model = CatBoostClassifier(
    auto_class_weights='Balanced', 
    verbose=0, 
    random_state=42
)


# Option 2: RandomizedSearchCV (plus rapide)
# On choisit d'optimiser le 'f1' (F1-Score) pour garder notre excellent compromis Precision/Recall
search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=20,             # Teste 20 combinaisons au hasard (augmentez si vous avez plus de temps)
    scoring='f1',          # La métrique cible à maximiser
    cv=3,                  # Cross-Validation à 3 plis
    n_jobs=-1,             # Utilise tous les cœurs de votre processeur
    random_state=42,
    verbose=1              # Affiche une barre de progression textuelle
)

# Lancer la recherche sur les données d'entraînement (ça peut prendre quelques minutes...)
print("Lancement de la recherche d'hyperparamètres...")
search.fit(X_train, y_train)

print("\n Recherche terminée !")
print(f"Meilleurs paramètres: {search.best_params_}")
print(f"Meilleur score CV (F1-Score): {search.best_score_:.4f}")

Lancement de la recherche d'hyperparamètres...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

 Recherche terminée !
Meilleurs paramètres: {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 300, 'depth': 6, 'auto_class_weights': 'Balanced'}
Meilleur score CV (F1-Score): 0.9111


#### Calcul de la probabilité RandomizedSearchCV

Si vous avez utilisé RandomizedSearchCV avec `n_iter=n`:

Formule: `P(top 5%) = 1 - (0.95)^n`

Exemple: n=100 → P ≈ 99.4%

In [69]:
# TODO: Si RandomizedSearchCV, calculer la probabilité
n_iter = 20  # Le nombre d'itérations que nous avons configuré précédemment
prob_top_5 = 1 - (0.95 ** n_iter)

print(f"Probabilité de trouver une combinaison dans le top 5% : {prob_top_5:.2%}")

Probabilité de trouver une combinaison dans le top 5% : 64.15%


### Mission 2.3.2 - Évaluation du Modèle Optimisé

In [70]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# TODO: Récupérer le meilleur modèle et évaluer sur test set
best_model = search.best_estimator_

# Prédictions des classes (0 ou 1)
y_pred_best = best_model.predict(X_test)

# Prédictions des probabilités (Nécessaire pour calculer les AUC)
# On récupère la colonne [:, 1] qui correspond à la probabilité d'appartenir à la classe 1 (Churn)
if hasattr(best_model, "predict_proba"):
    y_proba_best = best_model.predict_proba(X_test)[:, 1]
else:
    y_proba_best = best_model.decision_function(X_test)

print("\ PERFORMANCES DU MODÈLE OPTIMISÉ (TEST SET) :")
print("=" * 55)
print(classification_report(y_test, y_pred_best, target_names=['Fidèles (0)', 'Churners (1)']))

print("-" * 55)
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_proba_best):.4f}")
print("=" * 55)

\ PERFORMANCES DU MODÈLE OPTIMISÉ (TEST SET) :
              precision    recall  f1-score   support

 Fidèles (0)       0.98      1.00      0.99       570
Churners (1)       1.00      0.88      0.93        97

    accuracy                           0.98       667
   macro avg       0.99      0.94      0.96       667
weighted avg       0.98      0.98      0.98       667

-------------------------------------------------------
ROC-AUC : 0.9585
PR-AUC  : 0.9299


In [ ]:
# TODO: Vérifier l'objectif business: Recall ≥ 75%
recall_achieved = recall_score(y_test, y_pred_best)
objectif_recall = 0.75

if recall_achieved >= objectif_recall:
    print(f"✓ OBJECTIF ATTEINT: Recall = {recall_achieved:.2%} ≥ {objectif_recall:.0%}")
else:
    print(f"✗ Objectif non atteint: Recall = {recall_achieved:.2%} < {objectif_recall:.0%}")
    print("Actions: Ajuster le seuil, tester d'autres stratégies")

### Mission 2.3.3 - Optimisation XGBoost avec scale_pos_weight (Optionnel)

In [ ]:
# TODO: Si vous utilisez XGBoost, calculer et tester scale_pos_weight
# Formule: scale_pos_weight = count(classe négative) / count(classe positive)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Entraîner XGBoost avec scale_pos_weight
...

## PARTIE 2.4 - ANALYSE AVANCÉE (1h)

### Mission 2.4.1 - Learning Curves

In [ ]:
# TODO: Tracer les courbes d'apprentissage pour détecter overfitting/underfitting
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train,
    cv=5, scoring='recall',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

# Visualisation
...

**Question**: Y a-t-il de l'overfitting ? Plus de données améliorerait-elles le modèle ?

**Réponse**: ...

### Mission 2.4.2 - Calibration des Probabilités

**Formule Platt Scaling**:

`P_calibrated = 1 / (1 + exp(A × log(p/(1-p)) + B))`

où A et B sont appris par régression logistique sur les probabilités brutes.

In [ ]:
# TODO: Vérifier la calibration du modèle
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

prob_true, prob_pred = calibration_curve(y_test, y_proba_best, n_bins=10)

# Visualisation
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Model')
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
plt.xlabel('Predicted probability')
plt.ylabel('True probability')
plt.title('Calibration Curve')
plt.legend()
plt.show()

In [ ]:
# TODO: Si nécessaire, calibrer le modèle
# calibrated_model = CalibratedClassifierCV(best_model, method='sigmoid', cv=5)
# calibrated_model.fit(X_train, y_train)

...

## SAUVEGARDE DU MEILLEUR MODÈLE

In [ ]:
# TODO: Sauvegarder le meilleur modèle optimisé
import joblib

joblib.dump(best_model, '../models/best_model.pkl')

# Sauvegarder aussi les métadonnées
metadata = {
    'model_name': type(best_model).__name__,
    'best_params': search.best_params_,
    'recall': recall_achieved,
    'roc_auc': roc_auc_score(y_test, y_proba_best),
    'features': list(X_train.columns)
}

joblib.dump(metadata, '../models/metadata.pkl')

print("Modèle sauvegardé avec succès!")

## CONCLUSION JOUR 2

**Résumé des réalisations**:
- Nombre de modèles testés: ...
- Meilleur modèle: ...
- Stratégie de rééquilibrage: ...
- Performances finales:
  - Recall: ... (Objectif: ≥75%)
  - Precision: ...
  - ROC-AUC: ...

**Prochaines étapes (Jour 3)**:
- Explicabilité (SHAP, Feature Importance)
- API REST (Flask/FastAPI)
- Conteneurisation (Docker)
- Monitoring (MLflow, drift detection)